# Hierarchical Graph Generator: Multi-Level ZINC Generation

One Edge Generator creates the top-level interpretation graph. Two conditional stages then generate the base molecule.

In [1]:
from pathlib import Path
import runpy

roots = (Path.cwd(), *Path.cwd().parents)
candidates = [
    root / relative
    for root in roots
    for relative in (
        "notebooks/_bootstrap.py",
        "repos/abstractgraph-generative/notebooks/_bootstrap.py",
    )
]
bootstrap_path = next((path for path in candidates if path.is_file()), None)
if bootstrap_path is None:
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")
runpy.run_path(str(bootstrap_path))

{'__name__': '<run_path>',
 '__doc__': 'Shared bootstrap for notebooks in the AbstractGraph ecosystem.\n\nThis file is intended to be executed from notebooks via ``runpy.run_path``.\nIt resolves the current repo and workspace layout from a range of likely\nstarting directories, prepends available ``src`` directories to ``sys.path``,\nand normalizes the process working directory to the current repo root.\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/Users/f.costa/Code/abstractgraph-ecosystem/repos/abstractgraph-generative/notebooks/_bootstrap.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjec

In [2]:
from typing import cast

from sklearn.ensemble import RandomForestClassifier  # type: ignore[reportMissingModuleSource]

from abstractgraph.graphs import AbstractGraph
from abstractgraph.operators import (
    add,
    compose,
    cycle,
    intersection_edges,
    low_cut_partition,
    name,
    neighborhood,
    tree,
)
from abstractgraph.vectorize import AbstractGraphTransformer
from nsppk import NSPPK
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_ml.feasibility import (
    FeasibilityEstimator,
    FeasibilityEstimatorFeatureCannotExist,
)
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.edge_generator import EdgeGenerator
from abstractgraph_generative.graph_generator import GraphGenerator

## 1. Load a small molecule dataset

This example loads up to 300 ZINC molecules and skips records that cannot be parsed. The node-count limits keep the example focused on molecules with 30–50 atoms, so later graph operations stay manageable.

In [3]:
graphs, _ = ZINCLoader(on_error="skip").load(
    "zinc_250k",
    limit=300,
    min_node_count=30,
    max_node_count=50,
)
print(f"Loaded {len(graphs)} molecules")

Loaded 300 molecules


## 2. Define the graph hierarchy and generators

The two decomposition functions build progressively coarser views of each molecule: the first marks cycle and tree structure, and the second groups that result into small cuts. A conditional generator is assigned to each level. The edge generator handles the top-level graph, using a graph model and a feasibility check to guide its edge choices.

In [4]:
def one_hop_neighborhood(abstract_graph: AbstractGraph) -> AbstractGraph:
    return cast(AbstractGraph, neighborhood(abstract_graph, radius=1))


# Bottom-up: molecules -> cycle/tree graph -> coarser cut graph.
base_decomposition = compose(
    intersection_edges(),
    add(compose(name("cycle"), cycle()), compose(name("tree"), tree())),
)
coarse_decomposition = compose(
    intersection_edges(),
    compose(name("cut"), low_cut_partition(
        max_part_size=4,
        min_part_size=3,
        target_max_boundary_nodes=2,
        target_max_cut_edges=3,
        min_overlap_nodes=1,
        seed=7,
    )),
)

conditionals = [
    ConditionalAutoregressiveGenerator(decomposition_function=base_decomposition, nbits=14, base_cut_radius=0),
    ConditionalAutoregressiveGenerator(decomposition_function=coarse_decomposition, nbits=14, base_cut_radius=0),
]
feasibility = FeasibilityEstimator([
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=one_hop_neighborhood,
        nbits=19,
        parallel=False,
        n_jobs=1,
    )
])
vectorizer = cast(
    AbstractGraphTransformer,
    NSPPK(radius=1, distance=4, connector=0, nbits=14, dense=True),
)
edge_generator = EdgeGenerator(
    feasibility_estimator=feasibility,
    graph_estimator=GraphEstimator(
        transformer=vectorizer,
        estimator=RandomForestClassifier(
            n_estimators=80,
            class_weight="balanced_subsample",
            random_state=0,
            n_jobs=1,
        ),
    ),
    n_negative_per_positive=1,
    n_replicates=1,
    beam_size=3,
    max_restarts=1,
    fit_n_jobs=1,
    seed=0,
)
# 0: quiet expected retries, 1: normal warnings, 2: show every occurrence.
VERBOSE = 0
generator = GraphGenerator(
    edge_generator=edge_generator,
    conditional_generators=conditionals,
    verbose=VERBOSE,
    require_new_interpretation_graph=False,  # Keep a seed when edge generation cannot change its top-level interpretation.
)

## 3. Store the training graphs

`generator.store(graphs)` applies the configured hierarchy to the loaded molecules and builds the retrieval indexes used during sampling. This prepares examples at each level so the generator can find nearby graphs later.

In [5]:
generator.store(graphs)

## 4. Sample new molecules

Sampling starts from stored top-level graphs and uses nearby examples to guide top-level generation. The result is then expanded through the two conditional stages until base molecular graphs are produced. The final cell prints the number generated and draws the molecules for inspection.

This example relaxes anchor matching (`base_cut_radius=0`) and gives each conditional stage more search attempts. It also accepts the original top-level interpretation if edge generation cannot change it; this can improve completion while making some results less novel. The saved cell output below is from an earlier run; rerun the cell to evaluate these settings.

In [ ]:
samples = generator.sample(
    n_samples=3,  # Number of molecules to generate.
    n_interpretation_neighbors=40,  # Nearby top-level graphs used to guide interpretation generation; higher values add context and cost.
    n_conditional_neighbors=[40, 40],  # More retrieved examples give each descent stage a broader candidate pool.
    max_seed_attempts=60,  # Try more stored seeds before returning fewer than n_samples.
    conditional_generate_kwargs=[
        {"max_backtracks": 10000, "max_attempts_per_sample": 16},
        {"max_backtracks": 10000, "max_attempts_per_sample": 16},
    ],  # Give each conditional stage a larger search budget.
    random_state=247,  # Random seed for repeatable sampling choices.
)
print(f"Generated {len(samples)} molecules")
if samples:
    draw_molecules(samples, n_graphs_per_line=len(samples))